In [22]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from IPython.display import Image
from io import BytesIO
#from scrapy import fetch_website_links, fetch_website_contents
from openai import OpenAI

import requests
from bs4 import BeautifulSoup

In [16]:
url = "https://en.wikipedia.org/wiki/Albert_Einstein"
headers = {"User-Agent": "Mozilla/5.0"}  # Pretend to be a browser

#### Get all hyperlinks from a webpage

In [17]:
def redacted_links(url: str,headers):
    response = requests.get(url,headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    return [a['href'] for a in soup.find_all('a', href=True)]

links = redacted_links(url,headers)
print(len(links))

3911


## metadata

**A metadata field** is simply a named piece of information that describes a web page or document. In your scraping function, you’re collecting three common ones:

- **Title** → The text inside the <title> tag of the page. 
Example: "Albert Einstein - Wikipedia"

- **Description** → The content of <meta name="description" content="...">. 
This is often used by search engines to show a summary snippet. Example: "Learn about Albert Einstein, his life and contributions..." (if present).

- **Keywords** → The content of <meta name="keywords" content="...">. 
This used to be important for SEO but is now mostly ignored. Example: "physics, relativity, scientist" (rarely used today).

In [18]:
def redacted_metadata(url: str,headers):
    """Fetch common metadata (title, description, keywords) from the given URL."""
    response = requests.get(url,headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    metadata = {}
    # Page title
    metadata['title'] = soup.title.string if soup.title else None
    # Meta description
    desc_tag = soup.find('meta', attrs={'name':'description'})
    metadata['description'] = desc_tag['content'] if desc_tag else None
    # Meta keywords
    keywords_tag = soup.find('meta', attrs={'name':'keywords'})
    metadata['keywords'] = keywords_tag['content'] if keywords_tag else None
    return metadata

meta_data = redacted_metadata(url,headers)
print(len(meta_data))

md_text = f"""
**Title:** {meta_data['title']}
**Description:** {meta_data['description']}
**Keywords:** {meta_data['keywords']}
"""

display(Markdown(md_text))

3



**Title:** Albert Einstein - Wikipedia
**Description:** None
**Keywords:** None


## Get Image

In [34]:
def redacted_images(url: str, headers):
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    return [img['src'] for img in soup.find_all('img', src=True)]

images  = redacted_images(url,headers)
for img_url in images[5:3]:
    response = requests.get(img_url,headers=headers)
    img = Image.open(BytesIO(response.content))
    print(img.size)   # (width, height)

# Show first 3 images
for img_url in images[:9]:
    display(Image(url=img_url))

#### headings

In [37]:
def redacted_headings(url: str,headers):
    response = requests.get(url,headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    return [h.get_text(strip=True) for h in soup.find_all(['h1','h2','h3'])]

heading = redacted_headings(url,headers)
heading[:5]

['Contents',
 'Albert Einstein',
 'Life and career',
 'Childhood, youth and education',
 'Marriages, relationships and children']

In [ ]:
def redacted_contents(url: str, headers):
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    return [p.get_text(strip=True) for p in soup.find_all('p')]

content = redacted_contents(url,headers)
print(type(content))
md_text = "\n\n".join(content[:3]) # first 3 paragraphs
display(Markdown(md_text))

<class 'list'>




Albert Einstein[a](14 March 1879–18 April 1955) was a German-borntheoretical physicistbest known for developing thetheory of relativity. Einstein also made important contributions toquantum theory.[1][5]Hismass–energy equivalenceformulaE=mc2, which arises fromspecial relativity, has been called "the world's most famous equation".[6]He received the 1921Nobel Prize in Physicsfor "his services to theoretical physics, and especially for his discovery of the law of thephotoelectric effect".[7]

Born as a subject to theKingdom of Württemberg, part of theGerman Empire,[note 1]Einstein moved to Switzerland in 1895, forsaking his citizenship the following year. In 1896, at the age of seventeen, he enrolled in the mathematics and physics teaching diploma program at the Swissfederal polytechnic schoolin Zurich, graduating in 1900. He acquired Swiss citizenship a year later, which he kept for the rest of his life, and afterwards secured a permanent position at theSwiss Patent OfficeinBern. In 1905, he submitted a successfulPhDdissertation to theUniversity of Zurich. In 1914, he moved to Berlin to join thePrussian Academy of Sciencesand theHumboldt University of Berlin, becoming director of theKaiser Wilhelm Institute for Physicsin 1917; he also became aPrussianand consequently also German citizen again. In 1933, while Einstein was visiting the United States,Adolf Hitler came to powerin Germany. Horrified by theNazipersecution of his fellow Jews,[8]he decided to remain in the US, and was granted American citizenship in 1940.[9]On the eve ofWorld War II, he endorseda letterto PresidentFranklin D. Rooseveltalerting him to the potentialGerman nuclear weapons programand recommending that the US begin similar research, later carried out as theManhattan Project.